In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

chemin_racine = os.path.abspath('..')

if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

# 5. Mise en place du modèle XGBoost

Ici, nous avons pour but principal d'essayer de constituer un modèle XGBoost qui saura justifier de meilleures performances que la régression logistique qui nous sert de baseline.

## a. Import initial et setup de la cross-validation du modèle

Les modèles XGBoost étant des modèles boîte noire plus complexes qu'un modèle transparent comme une régression logistique, il est plus difficile de juger d'un surapprentissage ou d'un coup de chance. Il convient donc que l'on mette en place une validation croisée pour minimiser les chances de bruit causé par l'aléatoire et le surapprentissage. Nous choisissons d'utiliser une `StratifiedKFold` pour pouvoir garder le ratio bons payeurs / mauvais payeurs dans les splits. Puisque l'on avait choisi de prendre 20% du dataset en validation set, on va choisir de prendre 5 splits pour la cross-validation (on aura à chaque fois 4 splits pour train et le dernier pour test)

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

raw_data_dir = "../data/raw/"
raw_train_val_file = 'cs-training.csv'
raw_test_file = 'cs-test.csv'

credit_data = pd.read_csv(raw_data_dir + raw_train_val_file, index_col=0, header=0)
X = credit_data.drop("SeriousDlqin2yrs", axis=1)
y = credit_data["SeriousDlqin2yrs"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

## b. Benchmark de la baseline avec cross-validation :

Pour avoir une mesure totalement équitable, on fait le choix de remesurer les performances de la Régression Logistique qui nous sert de baseline en utilisant la même validation croisée que l'on utilisera pour les modèles XGBoost. Nous allons alors comparer les scores OOF de chacun pour avoir nos résultats. On rappelle que le score OOF consiste à grouper les évaluations faites sur les splits du dataset par les modèles en ayant uniquement connaissance des 4 autres splits.

On décide de faire une boucle manuelle pour les scores, établie dans une fonction dans `utils.py`.

Puisqu'on décide d'utiliser la baseline précédente ici, on en profite pour implémenter des fonctions qui font de l'import et de l'export de modèles en joblib.

In [ ]:
from src.utils import evaluate_model_oof, display_model_accuracy, load_pipeline

baseline_pipeline = load_pipeline("logreg_baseline_v2.joblib")

auc_roc_baseline, gini_baseline, oof_baseline = evaluate_model_oof(baseline_pipeline, X, y,skf)

display_model_accuracy(auc_roc_baseline,gini_baseline)


La baseline établie au notebook 4 avec notre approche OOP donne un **AUC-ROC à 0.856** et un **coefficient de Gini à 0.713** sur **l'intégralité du dataset** en OOF. Pour comparer, sur **20% du dataset** précédemment, on avait un **AUC-ROC à 0.858** et un **coefficient de Gini à 0.716**. On a donc des performances très similaires en score pur sur un nombre de clients beaucoup plus conséquent. Nous prendrons donc ces nouveaux chiffres comme référence pour la suite.

## c. Simple XGBoost

Dans cette section, nous allons commencer par entraîner et mesurer l'efficacité d'un modèle XGBoost sans fine tuner les paramètres pour se faire une idée de la performance de ce genre de modèles comparée à celle de notre baseline out of the box.



In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from src.pipelines import get_cleaner_pipeline

preprocessor = get_cleaner_pipeline()
simple_xgb = XGBClassifier(scale_pos_weight=(y == 0).sum() / (y == 1).sum(), eval_metric='auc', random_state=67)

simple_xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('simple_xgb', simple_xgb)
])

simple_xgb_auc_roc, simple_xgb_gini, simple_xgb_oof = evaluate_model_oof(simple_xgb_pipeline, X, y, skf)
display_model_accuracy(simple_xgb_auc_roc,simple_xgb_gini)

Notre modèle XGBoost out of the box affiche un **AUC-ROC à 0.848** et un **coefficient de Gini à 0.695 (soit 69.54%)**, ce qui constitue un recul de performance par rapport aux **0.856 d'AUC-ROC** et **71.29% de Gini** obtenus par notre baseline logistique en OOF. En effet, s'agissant d'un modèle boîte noire plus complexe qu'une régression linéaire transparente, l'algorithme est naturellement plus exposé au risque de surapprentissage lorsqu'il est appliqué sans fine tuner ses paramètres. Ce premier constat démontre qu'une architecture complexe brute ne surpasse pas automatiquement un modèle simple bien pré-traité et justifie pleinement le recours à notre cross-validation pour mener à bien l'optimisation des hyperparamètres nécessaire afin d'espérer battre notre socle de référence.

## d. Optimisation du modèle XGBoost :

Dans cette section, notre but va être de trouver les bons hyperparamètres qui vont permettre de réhausser les performances du modèle. Puisque nous avons identifié le surapprentissage comme une cause probable de la sous-performance du modèle XGBoost simple, notre recherche va s'articuler autour de sa minimisation. Voici les paramètres que nous allons fine tuner :
- `max_depth` : La profondeur maximale d'un arbre de décision au sein du modèle. Puisque nous suspectons du surapprentissage, le modèle par défaut a sûrement des arbres beaucoup trop profonds et donc beaucoup trop précis et focalisés sur des cas particuliers pour être bons dans une généralisation comportementale. Nous allons donc tester un set de valeurs plus ou moins faibles pour en avoir la certitude.
- `min_child_weight` : Un peu comme le paramètre précédent, augmenter celui-ci permet d'éviter que l'algorithme crée une règle uniquement pour quelques observations dans le dataset, évitant ainsi un surapprentissage sur des populations très précises et potentiellement non représentatives de la réalité.
- `learning_rate` : Le poids assigné à chaque nouvel arbre, ça dicte concrètement à quelle vitesse l'algorithme corrige ses erreurs.
- `n_estimators` : Nombre d'arbres qui sera construit dans l'algo du XGBoost. Il est optimisé en adéquation avec le learning rate pour donner suffisamment d'arbres pour laisser au modèle le temps de se corriger lorsque le learning rate est moins agressif.
- `colsample_bytree` : Définit le ratio des colonnes du dataset à fournir à chaque arbre du modèle. Concrètement, ça permet de forcer le modèle à faire des prédictions sans forcément se reposer sur les variables les plus parlantes comme `RevolvingUtilizationOfUnsecuredLines` et donc à mieux cerner les nuances des autres variables.
- `subsample` : Même esprit que le paramètre précédent, mais pour la proportion du set de lignes d'entraînement disponible pour chaque arbre.
- `reg_lambda` : Instaure une notion de prudence dans le modèle pour éviter les décisions radicales basées sur peu de signaux de défaut.

Le parcours de ces paramètres va se faire via `RandomizedSearchCV`

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'simple_xgb__max_depth': [3, 4, 5, 6, 7],
    'simple_xgb__min_child_weight': [10, 30, 50, 100],
    'simple_xgb__learning_rate': [0.03, 0.05, 0.08, 0.1],
    'simple_xgb__n_estimators': [150, 200, 300],
    'simple_xgb__colsample_bytree': [0.6, 0.7, 0.8],
    'simple_xgb__subsample': [0.7, 0.8, 0.9],
    'simple_xgb__reg_lambda': [1, 5, 10]
}

random_search = RandomizedSearchCV(
    estimator=simple_xgb_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=skf,
    random_state=67,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X, y)

best_auc = random_search.best_score_
best_gini = 2*best_auc - 1

display_model_accuracy(best_auc,best_gini)

print("Meilleurs paramètres :")
for param, value in random_search.best_params_.items():
    print(f"   - {param.replace('simple_xgb__', '')} : {value}")

best_xgb_pipeline = random_search.best_estimator_

### Copie de l'output de la cellule ci-dessus :
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Performances du modèle :
AUC-ROC : 0.867
Gini : 0.733 (Soit 73.30%)

Meilleurs paramètres :
   - subsample : 0.7
   - reg_lambda : 5
   - n_estimators : 150
   - min_child_weight : 50
   - max_depth : 5
   - learning_rate : 0.05
   - colsample_bytree : 0.6


## d. Analyse des résultats et de la configuration optimale

L'exécution de notre recherche d'hyperparamètres via `RandomizedSearchCV` sous validation croisée stratifiée en 5 folds a permis d'isoler la configuration la plus performante sur les 20 candidats testés.

Afin d'évaluer concrètement l'apport de cette étape d'optimisation et de déterminer si l'utilisation d'un algorithme non linéaire se justifie par rapport à notre baseline, nous pouvons résumer l'ensemble des scores Out-Of-Fold (OOF) obtenus jusqu'ici dans le tableau suivant :

### Comparaison des performances en validation croisée (OOF)

| Modèle | Prétraitement | AUC-ROC (OOF) | Coefficient de Gini | Évolution vs Baseline v2 |
| --- | --- | --- | --- | --- |
| **Régression Logistique v2** | Pipeline OOP + StandardScaler | 0.856 | 0.713 (71.29%)| *Référence linéaire* |
| **XGBoost Baseline (par défaut)** | Pipeline OOP brute | 0.848| 0.695 (69.54%)| -0.018 (-1.75%) |
| **XGBoost Optimisé (RandomSearchCV)** | Pipeline OOP brute | **0.867** | **0.733 (73.30%)** | **+0.020 (+2.01%)** |

### Interprétation des résultats et des hyperparamètres retenus

Le passage à 0.867 d'AUC-ROC et 73.30% de Gini constitue une amélioration nette de nos résultats. Nous pouvons tirer plusieurs conclusions quant au comportement de notre modèle et aux hyperparamètres sélectionnés :

* **L'efficacité du fine-tuning et de la non-linéarité :** Alors que la version par défaut de l'XGBoost affichait des performances inférieures à notre régression logistique en raison d'un surapprentissage (0.695 de Gini), la version optimisée parvient à surpasser le modèle linéaire (+2.01 points de Gini). Ce résultat indique que le jeu de données comporte des interactions complexes entre les variables que l'algorithme d'arbres de décision, une fois correctement régularisé, parvient à exploiter pour mieux distinguer les profils à risque.


* **Le contrôle du surapprentissage via la structure des arbres :** Les hyperparamètres retenus confirment la nécessité de contraindre la complexité du modèle sur des données financières bruitées. En limitant la profondeur maximale à 5 niveaux (`max_depth : 5`) et en imposant un poids minimum élevé de 50 observations par feuille (`min_child_weight : 50`), nous avons empêché l'algorithme de créer des règles de décision trop spécifiques sur des cas isolés. Cette régularisation structurelle explique directement le gain de performance observé sur la validation croisée.
* **L'apport de la régularisation et du sous-échantillonnage :** La présence d'une pénalisation L2 (`reg_lambda : 5`) a permis de lisser le poids des probabilités prédites, évitant ainsi au modèle de générer des scores de risque extrêmes sur de petits échantillons. De plus, le fait de restreindre l'accès à 60% des variables à chaque arbre (`colsample_bytree : 0.6`) a obligé l'algorithme à ne pas se baser uniquement sur les indicateurs de retards de paiement 90 jours (`NumberOfTimes90DaysLate`). Cela a contraint le modèle à accorder de l'importance à d'autres variables du dataset, valorisant ainsi le feature engineering réalisé sur la mensualité brute en euros (`MonthlyDebtAmount`).
* **La justification métier de la complexité :** Dans le cadre d'un modèle de Credit Scoring, le choix de remplacer un modèle linéaire simple et transparent par un algorithme plus complexe doit être justifié par un gain de performance significatif. Une progression de +2.01 points de Gini représente une amélioration concrète de la capacité de discrimination du modèle, ce qui permet de mieux identifier les risques de défaut tout en limitant les faux positifs.

Maintenant que nous disposons d'un modèle XGBoost optimisé et performant, la prochaine étape va consister à pallier son manque de transparence intrinsèque. Nous aborderons pour cela l'utilisation de la méthode SHAP afin de fournir une analyse d'explicabilité globale et locale, essentielle pour justifier les décisions du modèle dans un contexte bancaire.

### Soumission Kaggle :
Toujours dans une optique de comparaison, nous allons soumettre les prédictions de ce modèle à la compétition Kaggle [Give Me Some Credit](https://www.kaggle.com/competitions/GiveMeSomeCredit) pour avoir un référentiel de comparaison supplémentaire. Il est bon de rappeler que notre précédente soumission était faite avec la baseline Régression logistique v2 mais qui n'est pas passée par la cross validation.

In [ ]:
from src.kaggle_submissions import generate_kaggle_submission

test_credit_data = pd.read_csv(raw_data_dir + raw_test_file, index_col=0, header=0)
ids = test_credit_data.index
X_test = test_credit_data.drop(labels = ["SeriousDlqin2yrs"], axis = 1)

generate_kaggle_submission(best_xgb_pipeline,X_test, ids, "submission_xgb_fine_tuned.csv")

<br>
<p align="center">
  <img src="../images/kaggle_xgb_score.png" alt="Score Kaggle XGBoost" width="800"/>
</p>
<br>

La soumission sur la plateforme Kaggle nous donne un score AUC-ROC officiel de **0.86846** sur le jeu de test privé et de **0.86259** sur le jeu de test public.

### Comparaison des performances officielles (Kaggle)

| Métrique / Jeu de test                      | Baseline v2 (LogReg) | XGBoost Optimisé | Évolution            |
|---------------------------------------------|----------------------|------------------|----------------------|
| **Score Privé** *(Private Score)*           | 0.8574               | **0.8685**       | **+0.0111 (+1.29%)** |
| **Score Public** *(Public Score)*           | 0.8519               | **0.8626**       | **+0.0107 (+1.25%)** |
| **Score de Validation Locale** *(OOF / CV)* | 0.8580               | **0.8670**       | **+0.0090**          |

### Analyse des résultats de la soumission

* **Une progression confirmée sur le jeu d'évaluation final :** Sur le classement privé, qui constitue le véritable critère d'évaluation sur des données totalement invisibles, notre modèle XGBoost enregistre une hausse nette de **+0.0111 point d'AUC-ROC** par rapport à la régression logistique. Ce gain prouve que la non-linéarité et l'optimisation des hyperparamètres apportent un réel pouvoir discriminant supplémentaire par rapport au modèle linéaire.


* **La validation de notre méthodologie de cross-validation :** Notre score officiel sur le test privé (0.8685) s'avère extrêmement proche de notre score de validation croisée OOF obtenu localement (0.8670). Cette cohérence confirme que notre recherche d'hyperparamètres n'a pas provoqué de surapprentissage et que notre validation en 5 folds constitue un indicateur fiable et stable des performances réelles du modèle.